In [1]:
import imaplib
import email
from email.header import decode_header
import re
import pandas as pd

In [2]:
# Configuración
EMAIL = "diego.vaca.enriquez@gmail.com"
PASSWORD = "djol xhos knrg emhd"

# Palabras clave típicas de consumos
KEYWORDS = "Notificación de consumos"

In [3]:
# Conexión a Gmail
mail = imaplib.IMAP4_SSL("imap.gmail.com")
mail.login(EMAIL, PASSWORD)
mail.select("INBOX")

('OK', [b'38736'])

In [7]:
# Buscar todos los correos
status, messages = mail.search(None, 'SINCE "01-Jul-2026"')

resultados = []

for num in messages[0].split():
    status, data = mail.fetch(num, "(RFC822)")

    for response in data:
        if not isinstance(response, tuple):
            continue

        msg = email.message_from_bytes(response[1])

        asunto = msg.get("Subject", "")

        try:
            decoded = decode_header(asunto)[0]
            if isinstance(decoded[0], bytes):
                asunto = decoded[0].decode(
                    decoded[1] if decoded[1] else "utf-8",
                    errors="ignore"
                )
        except:
            pass

        remitente = msg.get("From", "")
        fecha = msg.get("Date", "")

        cuerpo = ""

        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_type() == "text/plain":
                    try:
                        cuerpo += part.get_payload(decode=True).decode(
                            errors="ignore"
                        )
                    except:
                        pass
        else:
            try:
                cuerpo = msg.get_payload(decode=True).decode(errors="ignore")
            except:
                pass

        texto = f"{asunto}\n{cuerpo}".lower()

        if any(k.lower() in texto for k in KEYWORDS):
            resultados.append({
                "fecha": fecha,
                "remitente": remitente,
                "asunto": asunto
            })

df = pd.DataFrame(resultados)

df.to_excel("consumos_tarjeta.xlsx", index=False)

print(f"Encontrados {len(df)} correos relacionados.")

abort: socket error: [WinError 10054] Se ha forzado la interrupción de una conexión existente por el host remoto